In [1]:
import pandas as pd
import json
import numpy as np

In [2]:
def process_qualt5_res_individual(_path: str, _cutoff: int, _readability_metric=''):
    _quality_df = pd.read_csv(_path)
    if('readability' in _path):
        _quality_df = _quality_df.rename(columns={'readability_score': 'quality'})
        _quality_df = _quality_df.query('readability_metric==@_readability_metric')
    else:
        _quality_df.quality = _quality_df.quality.apply(lambda x: np.exp(x))
    qualt5_individual_res = _quality_df.query('rank<@_cutoff')[['qid', 'docno', 'rank', 'quality']]

    
    qualt5_individual_res = qualt5_individual_res[qualt5_individual_res.quality != -1]
    qualt5_individual_res_groupby_qid = qualt5_individual_res.groupby(['qid'])['quality']
    _doc_qual_max = dict(zip(qualt5_individual_res_groupby_qid.max().index.astype('str').values, qualt5_individual_res_groupby_qid.max().values))
    _doc_qual_avg = dict(zip(qualt5_individual_res_groupby_qid.mean().index.astype('str').values, qualt5_individual_res_groupby_qid.mean().values))
    _doc_qual_min = dict(zip(qualt5_individual_res_groupby_qid.max().index.astype('str').values, qualt5_individual_res_groupby_qid.min().values))
    return _doc_qual_max, _doc_qual_avg, _doc_qual_min

def process_qualt5_res_integrated(_path: str, _readability_metric=''):
    _quality_df = pd.read_csv(_path)
    if('readability' in _path):
        _quality_df = _quality_df.rename(columns={'score': 'quality'})
        _quality_df = _quality_df.query('readability_metric==@_readability_metric')
    else:
        _quality_df.quality = _quality_df.quality.apply(lambda x: np.exp(x))
        
    qualt5_integrated_res = _quality_df[['qid', 'docno', 'quality']]

    _doc_qual_itg = dict(zip(qualt5_integrated_res.qid.astype('str').values, qualt5_integrated_res.quality.values))
    return _doc_qual_itg

In [25]:
import copy

_k = 5
_ret = 'e5'
_task = 'nq'
target_metric = 'f1'

if(_task=='nq'):
    _dataset_dev, _dataset_test, _prefix, _suffix = 'nq_dev', ['nq_test'], 'short', 'concise'
elif(_task=='dl'):
    _dataset_dev, _dataset_test, _prefix, _suffix = 'dev_small', ['19', '20'], 'random', 'prompt1'

In [26]:
# loading dev data
f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_1calls_0_0_bm25_dl_{_dataset_dev}_{_suffix}_eval.json')
zero_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}_eval.json')
k_evals = json.load(f)
f.close()

f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_1calls_1_0_{_ret}_dl_{_dataset_dev}_{_suffix}.json')
k_gens = json.load(f)
f.close()

f = open(f'../perplexity_eval/log_prob_temp_res/full_context_with_query/{_dataset_dev}_{_ret}_{_k}.json')
perpC = json.load(f)
f.close()

f = open(f'../perplexity_eval/log_prob_temp_res/individual_with_query/{_dataset_dev}_{_ret}_20.json')
perpC_indi = json.load(f)
f.close()
top_k_perpC_dict = {i[0]: [sub_i[1] for sub_i in i[1].items() if int(sub_i[0])<_k] for i in perpC_indi.items()}
avg_indi_perpC_dict = {i[0]:np.mean(i[1]) for i in top_k_perpC_dict.items()}
max_indi_perpC_dict = {i[0]:np.max(i[1]) for i in top_k_perpC_dict.items()}
min_indi_perpC_dict = {i[0]:np.min(i[1]) for i in top_k_perpC_dict.items()}

readability_res_path = f'../readability_eval/readability_res/individual_readability_{_dataset_dev}_{_ret}_top_10.csv'
readability_res_path_itg = f'../readability_eval/readability_res/integrated_readability_{_dataset_dev}_{_ret}_top_{_k}.csv'
available_metrics = pd.read_csv(readability_res_path).readability_metric.unique()
readability_max_dict, readability_min_dict, readability_avg_dict, readability_itg_dict = {}, {}, {}, {}
for _r_m in available_metrics:
    _r_m_max, _r_m_avg, _r_m_min = process_qualt5_res_individual(readability_res_path, _k, _r_m)
    _r_m_itg = process_qualt5_res_integrated(readability_res_path_itg, _r_m)
    readability_max_dict.update({_r_m: copy.deepcopy(_r_m_max)})
    readability_min_dict.update({_r_m: copy.deepcopy(_r_m_min)})
    readability_avg_dict.update({_r_m: copy.deepcopy(_r_m_avg)})
    readability_itg_dict.update({_r_m: copy.deepcopy(_r_m_itg)})

doc_qual_max, doc_qual_avg, doc_qual_min = process_qualt5_res_individual(f'../qualt5_eval/quality_res/{_ret}_{_dataset_dev}.csv', _k)
doc_qual_itg = process_qualt5_res_integrated(f'../qualt5_eval/quality_res/{_ret}_{_dataset_dev}_integrated_{_k}.csv')

qpp_df = pd.read_csv(f'./precomputed_qpps/{_ret}_{_k}_combined_qpp_{_dataset_dev}.csv')

dev_res = qpp_df[['qid', 'query']].drop_duplicates().copy()

# Expand the dataframe for the convenience of analysis
for qpp_name in qpp_df.qpp_method.unique():
    value_dict = dict(zip(qpp_df[qpp_df.qpp_method==qpp_name]['qid'], qpp_df[qpp_df.qpp_method==qpp_name]['qpp_estimate']))
    dev_res[qpp_name] = dev_res.qid.apply(lambda _qid: value_dict[_qid])

if(_task=='nq'):
    base_f1_dict_dev = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items()}
    base_em_dict_dev = {item[0]: item[1]['0']['0']['EM'] for item in zero_evals.items()}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items()}
    em_dict_dev = {item[0]: item[1]['0']['0']['EM'] for item in k_evals.items()}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items()}
elif(_task=='dl'):
    base_f1_dict_dev = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: item[1]['0']['0']['qrel_1']['f1']['max'] for item in k_evals.items() if ('0' in item[1].keys())}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

dev_res = dev_res[dev_res.qid.astype('str').isin(f1_dict.keys())]
dev_res['f1'] = dev_res.qid.apply(lambda x: f1_dict[str(x)])
dev_res['utility'] = dev_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict_dev[str(x)])
# dev_res['em'] = dev_res.qid.apply(lambda x: em_dict_dev[str(x)])
dev_res['prob(k)'] = dev_res.qid.apply(lambda x: kshot_prob_dict[str(x)])

dev_res['max(perpC)'] = dev_res.qid.apply(lambda x: max_indi_perpC_dict[str(x)])
dev_res['min(perpC)'] = dev_res.qid.apply(lambda x: min_indi_perpC_dict[str(x)])
dev_res['avg(perpC)'] = dev_res.qid.apply(lambda x: avg_indi_perpC_dict[str(x)])
dev_res['itg(perpC)'] = dev_res.qid.apply(lambda x: perpC[str(x)])

dev_res['max(docQual)'] = dev_res.qid.apply(lambda x: doc_qual_max[str(x)])
dev_res['min(docQual)'] = dev_res.qid.apply(lambda x: doc_qual_min[str(x)])
dev_res['avg(docQual)'] = dev_res.qid.apply(lambda x: doc_qual_avg[str(x)])
dev_res['itg(docQual)'] = dev_res.qid.apply(lambda x: doc_qual_itg[str(x)])
print(dev_res.shape)
dev_res = dev_res[dev_res.qid.astype('str').isin(readability_itg_dict['Spache'].keys())]
print(dev_res.shape)

print(dev_res.columns)

readability_cols = []
for _r_m in available_metrics:
    readability_cols += [f'max({_r_m})', f'min({_r_m})', f'avg({_r_m})', f'itg({_r_m})']
    print(_r_m, len(readability_max_dict[_r_m]))

    for qid in dev_res.qid.unique():
        if qid not in readability_max_dict[_r_m].keys():
            # print(qid)
            readability_max_dict[_r_m].update({str(qid): -1})
            readability_min_dict[_r_m].update({str(qid): -1})
            readability_avg_dict[_r_m].update({str(qid): -1})
        if qid not in readability_itg_dict[_r_m].keys():
            readability_itg_dict[_r_m].update({str(qid): -1})
    
    dev_res[f'max({_r_m})'] = dev_res.qid.apply(lambda x: readability_max_dict[_r_m][str(x)])
    dev_res[f'min({_r_m})'] = dev_res.qid.apply(lambda x: readability_min_dict[_r_m][str(x)])
    dev_res[f'avg({_r_m})'] = dev_res.qid.apply(lambda x: readability_avg_dict[_r_m][str(x)])
    dev_res[f'itg({_r_m})'] = dev_res.qid.apply(lambda x: readability_itg_dict[_r_m][str(x)])

dev_res = dev_res.drop(columns=[col for col in readability_cols if (dev_res[col] == -1).any()])
dev_res = dev_res.dropna(axis=1)

dev_res = dev_res.dropna(axis='index')
dev_res.head(3)
print(dev_res.columns)

(8757, 19)
(8757, 19)
Index(['qid', 'query', 'nqc', 'maxScore', 'spatial', 'a_ratio', 'bertQPP',
       'bertQPP(QV)', 'f1', 'utility', 'prob(k)', 'max(perpC)', 'min(perpC)',
       'avg(perpC)', 'itg(perpC)', 'max(docQual)', 'min(docQual)',
       'avg(docQual)', 'itg(docQual)'],
      dtype='object')
Dale Chall 8756
Spache 8756
Flesch-Kincaid 8756
Flesch 8756
Gunning Fog 8756
Coleman Liau 8756
ARI 8756
Linsear Write 8756
SMOG 0
Index(['qid', 'query', 'nqc', 'maxScore', 'spatial', 'a_ratio', 'bertQPP',
       'bertQPP(QV)', 'f1', 'utility', 'prob(k)', 'max(perpC)', 'min(perpC)',
       'avg(perpC)', 'itg(perpC)', 'max(docQual)', 'min(docQual)',
       'avg(docQual)', 'itg(docQual)', 'itg(Dale Chall)', 'itg(Spache)',
       'itg(Flesch-Kincaid)', 'itg(Flesch)', 'itg(Gunning Fog)',
       'itg(Coleman Liau)', 'itg(ARI)', 'itg(Linsear Write)'],
      dtype='object')


In [27]:
# loading test data

import numpy as np

def tool_for_aggregating_dl_performance(x):
    # the same as performLoader
    scores = []
    for answer_eval in x[1]['0'].values():
        scores.append(max(answer_eval['qrel_2']['f1']['max'], answer_eval['qrel_3']['f1']['max']))
    return np.mean(scores)

zero_evals, k_evals, k_gens, perpC  = {}, {}, {}, {}

_calls = 5 if _task=='dl' else 1

for _d in _dataset_test:
    f = open(f'../../rag_utility/eval_results/{_prefix}_answers_0shot_{_calls}calls_0_0_bm25_dl_{_d}_{_suffix}_eval.json')
    zero_evals.update(json.load(f))
    f.close()
    
    f = open(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_{_calls}calls_1_0_{_ret}_dl_{_d}_{_suffix}_eval.json')
    print(f'../../rag_utility/eval_results/{_prefix}_answers_{_k}shot_{_calls}calls_1_0_{_ret}_dl_{_d}_{_suffix}_eval.json')
    k_evals.update(json.load(f))
    f.close()
    
    f = open(f'../../rag_utility/gen_results/{_prefix}_answers_{_k}shot_{_calls}calls_1_0_{_ret}_dl_{_d}_{_suffix}.json')
    k_gens.update(json.load(f))
    f.close()

if(_task == 'dl'):
    _d = 'dl'
else:
    _d = 'nq_test'
    
f = open(f'../perplexity_eval/log_prob_temp_res/full_context_with_query/{_d}_{_ret}_{_k}.json')
perpC.update(json.load(f))
f.close()

f = open(f'../perplexity_eval/log_prob_temp_res/individual_with_query/{_d}_{_ret}_20.json')
perpC_indi = json.load(f)
f.close()
top_k_perpC_dict = {i[0]: [sub_i[1] for sub_i in i[1].items() if int(sub_i[0])<_k] for i in perpC_indi.items()}
avg_indi_perpC_dict = {i[0]:np.mean(i[1]) for i in top_k_perpC_dict.items()}
max_indi_perpC_dict = {i[0]:np.max(i[1]) for i in top_k_perpC_dict.items()}
min_indi_perpC_dict = {i[0]:np.min(i[1]) for i in top_k_perpC_dict.items()}

readability_res_path = f'../readability_eval/readability_res/individual_readability_{_d}_{_ret}_top_10.csv'
readability_res_path_itg = f'../readability_eval/readability_res/integrated_readability_{_d}_{_ret}_top_{_k}.csv'
available_metrics = pd.read_csv(readability_res_path).readability_metric.unique()
readability_max_dict, readability_avg_dict, readability_itg_dict, readability_min_dict = {}, {}, {}, {}
for _r_m in available_metrics:
    _r_m_max, _r_m_avg, _r_m_min = process_qualt5_res_individual(readability_res_path, _k, _r_m)
    _r_m_itg = process_qualt5_res_integrated(readability_res_path_itg, _r_m)
    readability_max_dict.update({_r_m: copy.deepcopy(_r_m_max)})
    readability_min_dict.update({_r_m: copy.deepcopy(_r_m_min)})
    readability_avg_dict.update({_r_m: copy.deepcopy(_r_m_avg)})
    readability_itg_dict.update({_r_m: copy.deepcopy(_r_m_itg)})

doc_qual_max, doc_qual_avg, doc_qual_min = process_qualt5_res_individual(f'../qualt5_eval/quality_res/{_ret}_{_d}.csv', _k)
doc_qual_itg = process_qualt5_res_integrated(f'../qualt5_eval/quality_res/{_ret}_{_d}_integrated_{_k}.csv')

qpp_df = pd.read_csv(f'./precomputed_qpps/{_ret}_{_k}_combined_qpp_{_d}.csv')

test_res = qpp_df[['qid', 'query']].drop_duplicates().copy()

# Expand the dataframe for the convenience of analysis
for qpp_name in qpp_df.qpp_method.unique():
    value_dict = dict(zip(qpp_df[qpp_df.qpp_method==qpp_name]['qid'], qpp_df[qpp_df.qpp_method==qpp_name]['qpp_estimate']))
    test_res[qpp_name] = test_res.qid.apply(lambda _qid: value_dict[_qid])

if(_task=='nq'):
    base_f1_dict_test = {item[0]: item[1]['0']['0']['F1'] for item in zero_evals.items()}
    base_em_dict_test = {item[0]: item[1]['0']['0']['EM'] for item in zero_evals.items()}
    f1_dict = {item[0]: item[1]['0']['0']['F1'] for item in k_evals.items()}
    em_dict_test = {item[0]: item[1]['0']['0']['EM'] for item in k_evals.items()}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items()}
elif(_task=='dl'):
    base_f1_dict_test = {item[0]: tool_for_aggregating_dl_performance(item) for item in zero_evals.items() if ('0' in item[1].keys())}
    f1_dict = {item[0]: tool_for_aggregating_dl_performance(item) for item in k_evals.items() if ('0' in item[1].keys())}
    kshot_prob_dict = {item[0]: np.mean(eval(item[1]['0']['0']['probs'])) for item in k_gens.items() if ('0' in item[1].keys())}

test_res = test_res[test_res.qid.astype('str').isin(f1_dict.keys())]
test_res['f1'] = test_res.qid.apply(lambda x: f1_dict[str(x)])
test_res['utility'] = test_res.qid.apply(lambda x: f1_dict[str(x)]-base_f1_dict_test[str(x)])
# test_res['em'] = test_res.qid.apply(lambda x: em_dict_test[str(x)])
test_res['prob(k)'] = test_res.qid.apply(lambda x: kshot_prob_dict[str(x)])

test_res['max(perpC)'] = test_res.qid.apply(lambda x: max_indi_perpC_dict[str(x)])
test_res['min(perpC)'] = test_res.qid.apply(lambda x: min_indi_perpC_dict[str(x)])
test_res['avg(perpC)'] = test_res.qid.apply(lambda x: avg_indi_perpC_dict[str(x)])
test_res['itg(perpC)'] = test_res.qid.apply(lambda x: perpC[str(x)])

test_res['max(docQual)'] = test_res.qid.apply(lambda x: doc_qual_max[str(x)])
test_res['min(docQual)'] = test_res.qid.apply(lambda x: doc_qual_min[str(x)])
test_res['avg(docQual)'] = test_res.qid.apply(lambda x: doc_qual_avg[str(x)])
test_res['itg(docQual)'] = test_res.qid.apply(lambda x: doc_qual_itg[str(x)])

print(test_res.shape)
test_res = test_res[test_res.qid.astype('str').isin(readability_itg_dict['Spache'].keys())]
print(test_res.shape)

readability_cols = []
for _r_m in available_metrics:
    readability_cols += [f'max({_r_m})', f'min({_r_m})', f'avg({_r_m})', f'itg({_r_m})']
    print(_r_m, len(readability_max_dict[_r_m]))

    for qid in test_res.qid.unique():
        if qid not in readability_max_dict[_r_m].keys():
            # print(qid)
            readability_max_dict[_r_m].update({str(qid): -1})
            readability_min_dict[_r_m].update({str(qid): -1})
            readability_avg_dict[_r_m].update({str(qid): -1})
        if qid not in readability_itg_dict[_r_m].keys():
            readability_itg_dict[_r_m].update({str(qid): -1})
    
    test_res[f'max({_r_m})'] = test_res.qid.apply(lambda x: readability_max_dict[_r_m][str(x)])
    test_res[f'min({_r_m})'] = test_res.qid.apply(lambda x: readability_min_dict[_r_m][str(x)])
    test_res[f'avg({_r_m})'] = test_res.qid.apply(lambda x: readability_avg_dict[_r_m][str(x)])
    test_res[f'itg({_r_m})'] = test_res.qid.apply(lambda x: readability_itg_dict[_r_m][str(x)])

dropped_columns = [col for col in readability_cols if (test_res[col] == -1).any()]
test_res = test_res.drop(columns=[col for col in readability_cols if (test_res[col] == -1).any()])
test_res = test_res.dropna(axis=1)
readability_cols = [e for e in readability_cols if e not in dropped_columns]
test_res.head(3).columns

../../rag_utility/eval_results/short_answers_5shot_1calls_1_0_e5_dl_nq_test_concise_eval.json
(3610, 19)
(3610, 19)
Dale Chall 3610
Spache 3610
Flesch-Kincaid 3610
Flesch 3610
Gunning Fog 3610
Coleman Liau 3610
ARI 3610
Linsear Write 3610
SMOG 0


Index(['qid', 'query', 'nqc', 'maxScore', 'spatial', 'a_ratio', 'bertQPP',
       'bertQPP(QV)', 'f1', 'utility', 'prob(k)', 'max(perpC)', 'min(perpC)',
       'avg(perpC)', 'itg(perpC)', 'max(docQual)', 'min(docQual)',
       'avg(docQual)', 'itg(docQual)', 'max(Dale Chall)', 'min(Dale Chall)',
       'avg(Dale Chall)', 'itg(Dale Chall)', 'max(Spache)', 'min(Spache)',
       'avg(Spache)', 'itg(Spache)', 'max(Flesch-Kincaid)',
       'min(Flesch-Kincaid)', 'avg(Flesch-Kincaid)', 'itg(Flesch-Kincaid)',
       'max(Flesch)', 'min(Flesch)', 'avg(Flesch)', 'itg(Flesch)',
       'max(Gunning Fog)', 'min(Gunning Fog)', 'avg(Gunning Fog)',
       'itg(Gunning Fog)', 'max(Coleman Liau)', 'min(Coleman Liau)',
       'avg(Coleman Liau)', 'itg(Coleman Liau)', 'max(ARI)', 'min(ARI)',
       'avg(ARI)', 'itg(ARI)', 'max(Linsear Write)', 'min(Linsear Write)',
       'avg(Linsear Write)', 'itg(Linsear Write)'],
      dtype='object')

In [28]:
from scipy import stats

In [29]:
import pandas as pd

corr = test_res.drop(columns=['qid', 'query', 'f1', 'utility', 'bertQPP(QV)']).corr(method="spearman")

df_content = []
for predictor_name in corr:
    df_content.append([predictor_name, stats.spearmanr(test_res[predictor_name], test_res[target_metric])[0], stats.kendalltau(test_res[predictor_name], test_res[target_metric])[0]])

a1 = pd.DataFrame(df_content, columns=['QPP_Method', 'Spearman', 'Kendall'])
a1.Spearman = a1.Spearman.apply(lambda x: round(x, 4))
a1.Kendall = a1.Kendall.apply(lambda x: round(x, 4))
a1.to_csv('./temp_for_pasting_results/a1.csv', index=False)
a1

,QPP_Method,Spearman,Kendall
0,nqc,0.1693,0.1276
1,maxScore,0.1792,0.1346
2,spatial,0.1484,0.1112
3,a_ratio,0.2093,0.1584
4,bertQPP,0.1291,0.0972
5,prob(k),0.3048,0.2289
6,max(perpC),0.0656,0.0497
7,min(perpC),0.0481,0.0364
8,avg(perpC),0.0686,0.0519
9,itg(perpC),0.0922,0.0696


#### Learned Combination -> Linear Regression

In [30]:
# keep dev and test data having the same features
common_cols = test_res.columns.intersection(dev_res.columns)

test_res = test_res[common_cols]
dev_res = dev_res[common_cols]

In [34]:
common_cols

Index(['qid', 'query', 'nqc', 'maxScore', 'spatial', 'a_ratio', 'bertQPP',
       'bertQPP(QV)', 'f1', 'utility', 'prob(k)', 'max(perpC)', 'min(perpC)',
       'avg(perpC)', 'itg(perpC)', 'max(docQual)', 'min(docQual)',
       'avg(docQual)', 'itg(docQual)', 'itg(Dale Chall)', 'itg(Spache)',
       'itg(Flesch-Kincaid)', 'itg(Flesch)', 'itg(Gunning Fog)',
       'itg(Coleman Liau)', 'itg(ARI)', 'itg(Linsear Write)'],
      dtype='object')

In [31]:
from sklearn import linear_model
import itertools
import pathlib

output_content = []

output_path = "./ecir_res/union_output_v1.csv"

# all_ablations = ['0', '1', '2', '3', '01', '02', '03', '12', '13', '23', '012', '013', '023', '123', '0123']
all_ablations = ['0123']

# define features
for use_postgen, pregen_combination in itertools.product([False], all_ablations):
    
    print(pregen_combination, use_postgen)
    used_qpp_methods = ['nqc', 'spatial', 'maxScore', 'a_ratio', 'bertQPP']

    used_predictors_unchanged, used_predictors_need_log = [], []
    # used_reader_centric_predictors = []
    if('0' in pregen_combination):
        used_predictors_need_log += used_qpp_methods
    if('1' in pregen_combination):
        used_predictors_unchanged += ['max(perpC)', 'avg(perpC)', 'itg(perpC)']
    if('2' in pregen_combination):
        used_predictors_need_log += ['max(docQual)', 'avg(docQual)', 'itg(docQual)']
    if('3' in pregen_combination):
        used_predictors_unchanged += readability_cols

    # if(target_metric=='utility'):
    #     dev_res = pd.concat([dev_res[dev_res.utility<0], dev_res[dev_res.utility>0.5]])

    # print(used_predictors_unchanged)
    # print(used_predictors_need_log)
    # load data
    dev_data_0, dev_data_1, test_data_0, test_data_1 = 0, 0, 0, 0
    if(used_predictors_unchanged != []):
        dev_data_0 = dev_res[used_predictors_unchanged]
        test_data_0 = test_res[used_predictors_unchanged]
    if(used_predictors_need_log != []):
        dev_data_1 = dev_res[used_predictors_need_log].apply(lambda x: np.log(1+x))
        test_data_1 = test_res[used_predictors_need_log].apply(lambda x: np.log(1+x))

    if(used_predictors_unchanged == []):
        dev_data, test_data = dev_data_1, test_data_1
    elif(used_predictors_need_log == []):
        dev_data, test_data = dev_data_0, test_data_0
    else:
        dev_data, test_data = np.hstack((dev_data_0, dev_data_1)), np.hstack((test_data_0, test_data_1))
        
    if(use_postgen):
        dev_data = np.hstack((dev_data, dev_res[['prob(k)']].values))
        test_data = np.hstack((test_data, test_res[['prob(k)']].values))

    # best before combination
    best_row = a1[a1.QPP_Method.isin(used_predictors_unchanged+used_predictors_need_log+use_postgen*['prob(k)'])].query('Spearman==Spearman.max()').iloc[0]
    print(f'Best before combination is {best_row.QPP_Method}, rho={best_row.Spearman}, tau={best_row.Kendall}')
    
    # linear regression
    reg = linear_model.LinearRegression()

    reg.fit(dev_data, dev_res[target_metric].values)
    coefs = reg.coef_
    intercept = reg.intercept_
    predictions = (dev_data * coefs).sum(axis=1) + intercept
    
    # print(coefs)
    # print(intercept)
    # print('accuracy on Dev set', stats.spearmanr(predictions, dev_res[target_metric]))
    
    try:
        f = open('./temp_for_pasting_results/weights.json', 'r+', encoding='UTF-8')
        weights = json.load(f)
        f.close()
    except:
        weights = {}
    
    weights.update({f'{_k}-{_ret}-{_task}-{target_metric}': list(coefs)})
    f = open('./temp_for_pasting_results/weights.json', 'w+')
    json.dump(weights, f, indent=4)
    f.close()
    
    predictions_test = ((test_data * coefs).sum(axis=1) + intercept)

    result_rho, result_tau = stats.spearmanr(predictions_test, test_res[target_metric])[0], stats.kendalltau(predictions_test, test_res[target_metric])[0]
    print(f'Accuracy on Test set, rho={result_rho}, tau={result_tau}')
    output_content.append(['GPP' if target_metric=='f1' else 'RPP', _task, _ret, _k, use_postgen, pregen_combination, result_rho, result_tau, best_row.QPP_Method, best_row.Spearman, best_row.Kendall])

    
    
temp_output = pd.DataFrame(output_content, columns=['Prediction Name', 'QA Task', 'Retriever', 'Top-Retrieved Docs', 'Use_Postgen', 'Combination_Number', 'Rho', 'Tau', 'Best Single Signal', 'Best Single Rho', 'Best Single Tau'])

temp_output

0123 False


KeyError: "['max(Dale Chall)', 'min(Dale Chall)', 'avg(Dale Chall)', 'max(Spache)', 'min(Spache)', 'avg(Spache)', 'max(Flesch-Kincaid)', 'min(Flesch-Kincaid)', 'avg(Flesch-Kincaid)', 'max(Flesch)', 'min(Flesch)', 'avg(Flesch)', 'max(Gunning Fog)', 'min(Gunning Fog)', 'avg(Gunning Fog)', 'max(Coleman Liau)', 'min(Coleman Liau)', 'avg(Coleman Liau)', 'max(ARI)', 'min(ARI)', 'avg(ARI)', 'max(Linsear Write)', 'min(Linsear Write)', 'avg(Linsear Write)'] not in index"

In [ ]:
predictions_dev = ((dev_data * coefs).sum(axis=1) + intercept)
predictions_test = ((test_data * coefs).sum(axis=1) + intercept)
stats.spearmanr(predictions_dev, dev_res[target_metric])

In [ ]:
analysis_dev = dev_res.copy()
analysis_dev['optimised_prediction'] = predictions_dev
analysis_dev['em'] = analysis_dev["qid"].apply(lambda x: em_dict_dev[x])
analysis_dev['zeroshot_em']  = analysis_dev["qid"].apply(lambda x: base_em_dict_dev[x])
analysis_dev['zeroshot_f1']  = analysis_dev["qid"].apply(lambda x: base_f1_dict_dev[x])

analysis_test = test_res.copy()
analysis_test['optimised_prediction'] = predictions_test
analysis_test['em'] = analysis_test["qid"].apply(lambda x: em_dict_test[x])
analysis_test['zeroshot_em']  = analysis_test["qid"].apply(lambda x: base_em_dict_test[x])
analysis_test['zeroshot_f1']  = analysis_test["qid"].apply(lambda x: base_f1_dict_test[x])

In [ ]:
print('em==1 avg(pred)=', analysis_dev[analysis_dev.em==1].optimised_prediction.mean())

In [ ]:
print('em==0 avg(pred)=', analysis_dev[analysis_dev.em==0].optimised_prediction.mean())

In [ ]:
analysis_test['optimised_prediction'].max()

## Find threshold by EM

In [ ]:
rank_of_last_0 = analysis_dev[analysis_dev['em']>0].shape[0]

In [ ]:
score_of_first_1 = analysis_dev[['optimised_prediction']].sort_values(by=['optimised_prediction'], ascending=True).iloc[rank_of_last_0].optimised_prediction
score_of_last_0 = analysis_dev[['optimised_prediction']].sort_values(by=['optimised_prediction'], ascending=True).iloc[rank_of_last_0-1].optimised_prediction
conformal_threshold = (score_of_last_0+score_of_first_1)/2
print(conformal_threshold)

predicted_negative_dev = analysis_dev[analysis_dev.optimised_prediction<=conformal_threshold]
predicted_positive_dev = analysis_dev[analysis_dev.optimised_prediction>conformal_threshold]

false_negative_num = predicted_negative_dev['em'].sum()
pred_negative_num = predicted_negative_dev.shape[0]
true_positive_num = predicted_positive_dev['em'].sum()
pred_positive_num = predicted_positive_dev.shape[0]
print(false_negative_num, '/', pred_negative_num, ';', true_positive_num, '/', pred_positive_num)

print('precision', true_positive_num/pred_positive_num)

print('wrongly predicted negative labels', false_negative_num/pred_negative_num)

print('accuracy', (pred_negative_num-false_negative_num+true_positive_num)/analysis_dev.shape[0])

print('recall', true_positive_num/analysis_dev.em.sum())

In [ ]:
predicted_negative_test = analysis_test[analysis_test.optimised_prediction<=conformal_threshold]
predicted_positive_test = analysis_test[analysis_test.optimised_prediction>conformal_threshold]

false_negative_num = predicted_negative_test['em'].sum()
pred_negative_num = predicted_negative_test.shape[0]
true_positive_num = predicted_positive_test['em'].sum()
pred_positive_num = predicted_positive_test.shape[0]
print(false_negative_num, '/', pred_negative_num, ';', true_positive_num, '/', pred_positive_num)

print('precision', true_positive_num/pred_positive_num)

print('wrongly predicted negative labels', false_negative_num/pred_negative_num)

print('accuracy', (pred_negative_num-false_negative_num+true_positive_num)/analysis_test.shape[0])

print('recall', true_positive_num/analysis_test.em.sum())

In [ ]:
predicted_negative_test[predicted_negative_test['f1']<predicted_negative_test['zeroshot_f1']].shape

In [ ]:
predicted_negative_test[predicted_negative_test['f1']>predicted_negative_test['zeroshot_f1']].shape

In [ ]:
predicted_negative_test.zeroshot_f1.sum()-predicted_negative_test.f1.sum()

## find threshold by the overall performance

In [ ]:
ranked_analysis_dev = analysis_dev[['f1', 'zeroshot_f1', 'optimised_prediction']].sort_values(by=['optimised_prediction'], ascending=True).copy()

total_gain = 0
highest_total_gain = -1
conformal_threshold = -1

for num, row in enumerate(ranked_analysis_dev.iterrows()):
    total_gain += (row[1].zeroshot_f1-row[1].f1)
    if(total_gain > highest_total_gain):
        highest_total_gain = total_gain
        conformal_threshold = row[1].optimised_prediction
        # print(num, total_gain, row[1].optimised_prediction)
print(conformal_threshold)

In [ ]:
predicted_negative_test = analysis_test[analysis_test.optimised_prediction<=conformal_threshold]
predicted_positive_test = analysis_test[analysis_test.optimised_prediction>conformal_threshold]

false_negative_num = predicted_negative_test['em'].sum()
pred_negative_num = predicted_negative_test.shape[0]
true_positive_num = predicted_positive_test['em'].sum()
pred_positive_num = predicted_positive_test.shape[0]
print(false_negative_num, '/', pred_negative_num, ';', true_positive_num, '/', pred_positive_num)

print('precision', true_positive_num/pred_positive_num)

print('wrongly predicted negative labels', false_negative_num/pred_negative_num)

print('accuracy', (pred_negative_num-false_negative_num+true_positive_num)/analysis_test.shape[0])

print('recall', true_positive_num/analysis_test.em.sum())

In [ ]:
predicted_negative_test[predicted_negative_test['f1']<predicted_negative_test['zeroshot_f1']].shape

In [ ]:
predicted_negative_test[predicted_negative_test['f1']>predicted_negative_test['zeroshot_f1']].shape

In [ ]:
predicted_negative_test.zeroshot_f1.sum()-predicted_negative_test.f1.sum()